# 🚢 Titanic Interactive Dashboard

Ein interaktives Streamlit-Dashboard zum Erkunden des Titanic-Datensatzes mit Echtzeit-Filtern.

**Technologien:** `streamlit`, `plotly`, `pandas`

In diesem Notebook wird der gesamte Code für das Dashboard vorgestellt und jede Komponente erklärt.

## 📋 Inhaltsverzeichnis

1. [Importe & Setup](#1--importe-setup)
2. [Daten laden & bereiten](#2--daten-laden-bereiten)
3. [Sidebar-Filter](#3--sidebar-filter)
4. [KPI-Karten (Summary Statistics)](#4--kpi-karten-summary-statistics)
5. [Interaktive Visualisierungen](#5--interaktive-visualisierungen)
6. [CSV-Export](#6--csv-export)
7. [Kompletter Dashboard-Code](#7--kompletter-dashboard-code)

---

## 1️⃣ Importe & Setup

Wir benötigen Streamlit für das Web-Framework, Plotly für interaktive Diagramme und Pandas für die Datenverarbeitung.

In [ ]:
# ==============================================
# KAPITEL 1: IMPORTE
# ==============================================

import streamlit as st
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from datetime import datetime

# Styling
st.set_page_config(page_title="Titanic Explorer", layout="wide")
print('Alle Importe erfolgreich geladen ✅')

---

## 2️⃣ Daten laden & bereiten

Der Titanic-Datensatz wird direkt von GitHub geladen und mit `@st.cache_data` zwischengespeichert, damit wiederholte Filtern schnell sind.

In [ ]:
# ==============================================
# KAPITEL 2: DATEN LADEN
# ==============================================

@st.cache_data
def load_data():
    url = 'https://raw.githubusercontent.com/datasciencehandbook/notebooks/data/titanic.csv'
    df = pd.read_csv(url)
    return df

# Daten laden
df = load_data()

# Datentypen bereinigen (Embarked als String)
df['Embarked'] = df['Embarked'].astype(str)
df['Sex'] = df['Sex'].str.lower()  # Normalisierung

print(f'✅ Datensatz geladen: {len(df)} Zeilen, {len(df.columns)} Spalten')
print()
print('Verfügbare Features:')
for col in df.columns:
    nulls = df[col].isnull().sum()
    uniq = df[col].nunique()
    print(f'  - {col:15s} ({df[col].dtype:12s}) | null={nulls:>3} | unique={uniq}')

# Beispiel-Daten anzeigen
with st.expander('📋 Beispieldaten (erste 5 Zeilen)', expanded=False):
    st.dataframe(df.head(), use_container_width=True)


---

## 3️⃣ Sidebar-Filter

Der Benutzer kann nach Passagierklasse, Geschlecht, Alter und Einboard-Port filtern. Alle Filter wirken sich live auf die Charts aus.

In [ ]:
# ==============================================
# KAPITEL 3: FILTER LOGIC
# ==============================================

# ---- Filter definieren ----
pclass_options = ['Alle'] + [str(i) for i in range(1, 4)]
sex_options = ['Alle'] + list(df['Sex'].unique())
embarked_options = ['Alle'] + list(df['Embarked'].dropna().unique())

min_age = int(df['Age'].dropna().min())
max_age = int(df['Age'].dropna().max())

# ---- Filter logik (ohne Streamlit-Widgets) ----
# In der echten App würden wir st.selectbox etc. verwenden.
# Hier simulieren wir mit festen Werten:

class_filter = 'Alle'  # Alle Klassen
sex_filter = 'Alle'    # Beide Geschlechter
embark_filter = 'Alle'
age_min, age_max = min_age, max_age

# ---- Daten filtern ----
filtered = df.copy()
if class_filter != 'Alle':
    filtered = filtered[filtered['Pclass'] == int(class_filter)]
if sex_filter != 'Alle':
    filtered = filtered[filtered['Sex'] == sex_filter]
filtered = filtered[(filtered['Age'] >= age_min) & (filtered['Age'] <= age_max)]
if embark_filter != 'Alle':
    filtered = filtered[filtered['Embarked'] == embark_filter]

print(f'📊 Gesamt: {len(df)} | Gefiltert: {len(filtered)}')
print(f'   Filter: Klasse={class_filter}, Geschlecht={sex_filter}, '
      f'Alter={age_min}-{age_max}, Port={embark_filter}')

# ---- Statistik der gefilterten Daten ----
print(f"\n=== STATISTIK (GEFILTERT) ===")
if len(filtered) > 0:
    print(f"Passagiere: {len(filtered)}")
    print(f"Überlebensrate: {filtered['Survived'].mean():.1%}")
    print(f"Ø Ticketpreis: ${filtered['Fare'].mean():.0f}")
    print(f"Ø Alter: {filtered['Age'].dropna().mean():.0f} Jahre")


---

## 4️⃣ KPI-Karten (Summary Statistics)

Zwei Zeilen mit vier Spalten pro Reihe für die wichtigsten Kennzahlen.

In [ ]:
# ==============================================
# KAPITEL 4: KPI KARTEN
# ==============================================

if len(filtered) > 0:
    print('=== ZUSAMFASSUNG ===')
    
    # Wir simulieren st.columns mit Print-Ausgabe
    pass_count = len(filtered)
    surv_rate = f"{filtered['Survived'].mean():.1%}"
    avg_fare = f"${filtered['Fare'].mean():.0f}"
    avg_age = f"{filtered['Age'].dropna().mean():.0f}" if len(filtered['Age'].dropna()) > 0 else 'N/A'

    # In Streamlit würde es so aussehen:
    # col1, col2, col3, col4 = st.columns(4)
    # with col1: st.metric('Passagiere', pass_count)
    # with col2: st.metric('Überlebensrate', surv_rate)
    # ...etc
    
    print(f"  Passagiere:     {pass_count:>10}")
    print(f"  Überlebensrate: {surv_rate:>10}")
    print(f"  Ø Ticketpreis:  {avg_fare:>10}")
    print(f"  Ø Alter (Jahre):{avg_age:>10}")

---

## 5️⃣ Interaktive Visualisierungen

Jedes Chart wird mit Plotly erstellt und kann interaktiv bedient werden (Zoom, Hover, Download als PNG).

In [ ]:
# ==============================================
# KAPITEL 5a: ÜBERLEBEN NACH KLASSE (PLOTLY)
# ==============================================

# Berechnen
surv_by_class = filtered.groupby('Pclass')['Survived'].agg(['mean', 'count']).rename(
    columns={'mean': 'rate', 'count': 'pax'})
surv_by_class['rate'] = (surv_by_class['rate'] * 100).round(1)

# Plotly Balkendiagramm
fig1 = px.bar(
    surv_by_class.reset_index(),
    x='Pclass', y='rate',
    title='Überlebensrate nach Passagierklasse (%)',
    color='Pclass',
    color_discrete_sequence=['#e74c3c', '#f39c12', '#2ecc71'],
    labels={'Pclass': 'Klasse', 'rate': 'Überlebensrate (%)'},
    text='rate'
)

fig1.update_layout(
    xaxis_title='',  # Klassen sind in x
    yaxis_title='Überlebensrate (%)',
    font=dict(size=12),
    bargap=0.3,
    height=400,
    template='plotly_white'
)

# Werte auf den Balken
fig1.update_traces(texttemplate='%{y:.1f}%', textposition='outside')

print('📊 Chart: Überlebensrate nach Klasse (Plotly)')
# In Streamlit würde es st.plotly_chart(fig1, use_container_width=True) machen
fig1.show()


In [ ]:
# ==============================================
# KAPITEL 5b: ALTERSVERTEILung (PLOTly)
# ==============================================

valid_ages = filtered[filtered['Age'].notna()]
if len(valid_ages) > 0:
    fig2 = px.histogram(
        valid_ages,
        x='Age',
        color='Survived',
        barmode='overlay',
        opacity=0.7,
        title='Altersverteilung — Überlebt vs. Gestorben',
        labels={'Age': 'Alter (Jahre)', 'Survived': 'Status'},
        color_discrete_map={1: '#2ecc71', 0: '#e74c3c'}
    )
    fig2.update_layout(
        xaxis_title='Alter (Jahre)',
        yaxis_title='Anzahl Passagiere',
        font=dict(size=12),
        height=400,
        template='plotly_white'
    )
    fig2.show()

# Zusätzlich: Überlebensrate nach Altersgruppe als Heatmap
print('\n📊 Chart: Überlebensrate nach Altersgruppe (Heatmap)')
age_groups = pd.cut(filtered['Age'], bins=[0, 5, 12, 18, 35, 60, 100],
                    labels=['0-5', '6-12', '13-18', '19-34', '35-59', '60+'])
if len(filtered) > 0 and age_groups.notna().any():
    pivot = filtered.copy()
    pivot['AgeGroup'] = age_groups
    pivot = pivot.dropna(subset=['AgeGroup', 'Survived'])
    
    if len(pivot) > 0:
        heat_data = pivot.pivot_table(
            index='AgeGroup', columns='Pclass', values='Survived', aggfunc='mean'
        )
        fig3 = px.imshow(
            heat_data.round(2) * 100,
            labels=dict(color='Überlebensrate (%)'),
            x=[f'Pclass {i}' for i in range(1, 4)],
            y=list(heat_data.index),
            title='Überlebensrate: Alter × Klasse (Heatmap)',
            color_continuous_scale='RdYlGn',
            text_auto='.1f'
        )
        fig3.update_layout(
            font=dict(size=12),
            height=400,
            template='plotly_white'
        )
        fig3.show()


In [ ]:
# ==============================================
# KAPITEL 5c: TICKETPREIS vs. ALTER (SCATTER)
# ==============================================

valid_fares = filtered[filtered['Fare'].notna()]
if len(valid_fares) > 0:
    fig4 = px.scatter(
        valid_fares,
        x='Age',
        y='Fare',
        color='Survived',
        size='Fare',
        opacity=0.7,
        title='Ticketpreis vs. Alter — Gefärbt nach Überlebensstatus',
        labels={'Age': 'Alter (Jahre)', 'Fare': 'Ticketpreis ($)', 'Survived': 'Status'},
        color_discrete_map={1: '#2ecc71', 0: '#e74c3c'}
    )
    fig4.update_layout(
        xaxis_title='Alter (Jahre)',
        yaxis_title='Ticketpreis ($)',
        font=dict(size=12),
        height=400,
        template='plotly_white'
    )
    fig4.show()


---

## 6️⃣ CSV-Export

Der Benutzer kann die gefilterten Daten als CSV herunterladen.


In [ ]:
# ==============================================
# KAPITEL 6: CSV EXPORT
# ==============================================

print('=== GEFILTERTE DATEN (Erste 10 Zeilen) ===')
print(filtered.head(10).to_string())
print(f'\nGesamt: {len(filtered)} Zeilen')

# In Streamlit würde es so aussehen:
# csv = filtered.to_csv(index=False).encode('utf-8')
# st.download_button('📥 CSV herunterladen', data=csv, file_name='titanic_filtered.csv', mime='text/csv')

---

## 7️⃣ Kompletter Dashboard-Code

Hier ist der vollständige Code für `dashboard_titanic.py`:


In [ ]:
# ==============================================
# KOMPLETTER DASHBOARD-CODE
# ==============================================

full_code = '''import streamlit as st
import pandas as pd
import plotly.express as px
from datetime import datetime

st.title("🚢 Titanic Explorer - Interactive Dashboard")
st.markdown("*An interactive dashboard for exploring the Titanic dataset.*")

@st.cache_data
def load_data():
    url = "https://raw.githubusercontent.com/datasciencehandbook/notebooks/data/titanic.csv"
    return pd.read_csv(url)

df = load_data()

# Sidebar Filters
with st.sidebar:
    st.header("Filters")
    pc = st.selectbox('Passenger Class', ['All'] + [str(i) for i in range(1,4)])
    sex = st.selectbox('Gender', ['All'] + list(df['Sex'].unique()))
    min_age, max_age = df['Age'].dropna().min(), df['Age'].dropna().max()
    age_range = st.slider("Age Range (years)", int(min_age), int(max_age), 
                          (int(min_age), int(max_age)))
    emb = st.selectbox('Embarkation Port', ['All'] + list(df['Embarked'].unique()))

# Apply Filters
filtered = df.copy()
if pc != 'All': filtered = filtered[filtered['Pclass'] == int(pc)]
if sex != 'All': filtered = filtered[filtered['Sex'] == sex]
filtered = filtered[(filtered['Age'] >= age_range[0]) & (filtered['Age'] <= age_range[1])]
if emb != 'All': filtered = filtered[filtered['Embarked'] == emb]

# KPI Cards
st.header("Summary Statistics")
c1, c2, c3, c4 = st.columns(4)
with c1: st.metric('Passengers', len(filtered))
with c2: st.metric('Survival Rate', f'{filtered["Survived"].mean():.1%}')
with c3: st.metric('Avg Fare ($)', f'${filtered["Fare"].mean():.0f}')
with c4: st.metric('Avg Age (yrs)', f'{filtered["Age"].dropna().mean():.0f}')

# Charts
st.header("Visualizations")
t1, t2, t3 = st.tabs(['Survival by Class', 'Age Distribution', 'Fare vs Age'])

with t1:
    s = filtered.groupby('Pclass')['Survived'].agg(['mean','count']).rename(columns={'mean':'rate','count':'pax'})
    s['rate'] = (s['rate']*100).round(1)
    f1 = px.bar(s.reset_index(), x='Pclass', y='rate', title='Survival Rate (%)', 
                color='Pclass', color_discrete_sequence=['#e74c3c','#f39c12','#2ecc71'])
    st.plotly_chart(f1, use_container_width=True)

with t2:
    f2 = px.histogram(filtered[filtered['Age'].notna()], x='Age', color='Survived', 
                      barmode='overlay', opacity=0.7, title='Age - Survived vs Died')
    st.plotly_chart(f2, use_container_width=True)

with t3:
    valid = filtered[filtered['Fare'].notna()]
    f3 = px.scatter(valid, x='Age', y='Fare', color='Survived', size='Fare', 
                    opacity=0.7, title='Fare vs Age', labels={'Age':'Age (years)', 'Fare': 'Fare ($)'})
    st.plotly_chart(f3, use_container_width=True)

# Data Table + Download
with st.expander('Raw Data (filtered)', expanded=False):
    st.dataframe(filtered.reset_index(drop=True), use_container_width=True)
    csv = filtered.to_csv(index=False).encode('utf-8')
    st.download_button('Download CSV', data=csv, file_name='titanic_filtered.csv', mime='text/csv')
'''

from IPython.display import Markdown
print(full_code)
# In einer echten Umgebung würde man den Code direkt in Streamlit ausführen:
# !streamlit run dashboard_titanic.py

---

## 🚀 Dashboard starten

Um das Dashboard lokal zu starten:

```bash
# 1. Abhängigkeiten installieren
pip install streamlit plotly pandas numpy seaborn

# 2. Dashboard starten
streamlit run projects/eda-titanic-dashboard/dashboard_titanic.py
```

Der Browser öffnet sich automatisch unter `http://localhost:8501`.